# Hyperliquid × Bitcoin Fear/Greed Index
### Trader Behavior & Performance Analysis - How Market Sentiment Drives Trader Behavior on Hyperliquid

## Data Understanding

**Business Problem:**  
Crypto traders operating on Hyperliquid make leveraged decisions in real time. Yet most traders ignore macro market sentiment — the prevailing Fear or Greed in the Bitcoin market — when entering or sizing trades. This leaves systematic, measurable edge on the table.

**Aggregate PnL alone does not reveal whether losses are caused by bad trades or bad timing relative to market sentiment.**

**Decision this analysis supports:**  
When should a trader on Hyperliquid increase or decrease leverage, trade frequency, and directional bias — based on the current Fear/Greed reading?

**Stakeholder:** Quantitative traders, DeFi risk managers, Hyperliquid protocol analysts.

**What metric defines success:**  
A measurable, statistically meaningful difference in PnL, win rate, or leverage behaviour across Fear vs Greed sentiment regimes.

**What happens if we do nothing:**  
Traders continue executing without sentiment context, leaving risk-adjusted return improvements uncaptured.

---

**Goal of This Notebook:** Build the analytical foundation before any cleaning or modeling. Answer three questions:
1. What data do we actually have?
2. Is it usable for sentiment-driven behavioral analysis?
3. What are the risks, biases, and alignment challenges in this dataset?

**Approach:**
```
Inspect Fear/Greed table → Inspect Trader table → Assess alignment
→ Document risks → Confirm viability for sentiment analysis
```

## Success Metrics

| Metric | Definition | Timeframe | Segments |
|---|---|---|---|
| Daily PnL per trader | Sum of `Closed PnL` per account per day | Daily | All accounts |
| Win rate | % of closing trades with PnL > 0 | Daily / per regime | Sentiment bucket |
| Avg trade size (USD) | Mean `Size USD` per day | Daily | Sentiment bucket |
| Leverage proxy | `Size USD / abs(Start Position)` where applicable | Per trade | High vs low leverage |
| Long/short ratio | Open Long count / Open Short count per day | Daily | Sentiment bucket |
| Trade frequency | Trades per account per day | Daily | Frequent vs infrequent |
| Drawdown proxy | Min cumulative PnL within a losing streak | Per account | Winners vs inconsistent |

---
## Dataset 1 — Bitcoin Fear/Greed Index

The Fear/Greed index is the macro sentiment signal. Published daily, it
scores Bitcoin market mood on a 0–100 scale. We inspect it first to
understand coverage, classification balance, and date alignment.

In [1]:
import pandas as pd
import numpy as np
import warnings
warnings.filterwarnings('ignore')

In [2]:
fg = pd.read_csv('fear_greed_index.csv')
fg['date'] = pd.to_datetime(fg['date'])
fg.head()

,timestamp,value,classification,date
0,1517463000,30,Fear,2018-02-01
1,1517549400,15,Extreme Fear,2018-02-02
2,1517635800,40,Fear,2018-02-03
3,1517722200,24,Extreme Fear,2018-02-04
4,1517808600,11,Extreme Fear,2018-02-05


### Dataset Structure

In [3]:
fg.shape

(2644, 4)

In [4]:
fg.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 2644 entries, 0 to 2643
Data columns (total 4 columns):
 #   Column          Non-Null Count  Dtype         
---  ------          --------------  -----         
 0   timestamp       2644 non-null   int64         
 1   value           2644 non-null   int64         
 2   classification  2644 non-null   object        
 3   date            2644 non-null   datetime64[ns]
dtypes: datetime64[ns](1), int64(2), object(1)
memory usage: 82.8+ KB


**Data Type Assessment — Fear/Greed Table:**

Four columns — `timestamp` (unix), `value` (0–100 integer), `classification` (string label),
and `date` (YYYY-MM-DD). All four are fully populated. No casting issues.
The `timestamp` column is redundant with `date` and will be dropped in cleaning.
The `value` column is the continuous underlying score; `classification` is the bucketed label —
both are useful for different analyses.

### Missing Values & Duplicates

In [5]:
fg.isnull().sum()

timestamp         0
value             0
classification    0
date              0
dtype: int64

In [6]:
fg.duplicated().sum()

np.int64(0)

**Missing Value Assessment — Fear/Greed Table:**

Zero nulls. Zero duplicates. This is a textbook clean time-series.
Every calendar date has exactly one sentiment reading — ideal for a daily join.

### Sentiment Distribution

In [7]:
fg['classification'].value_counts()

classification
Fear             781
Greed            633
Extreme Fear     508
Neutral          396
Extreme Greed    326
Name: count, dtype: int64

In [8]:
fg['classification'].value_counts(normalize=True).round(3)

classification
Fear             0.295
Greed            0.239
Extreme Fear     0.192
Neutral          0.150
Extreme Greed    0.123
Name: proportion, dtype: float64

In [9]:
fg['value'].describe()

count    2644.000000
mean       46.981089
std        21.827680
min         5.000000
25%        28.000000
50%        46.000000
75%        66.000000
max        95.000000
Name: value, dtype: float64

**Sentiment Distribution Assessment:**

| Classification | Count | % |
|---|---|---|
| Fear | 781 | 29.5% |
| Greed | 633 | 23.9% |
| Extreme Fear | 508 | 19.2% |
| Neutral | 396 | 15.0% |
| Extreme Greed | 326 | 12.3% |

The distribution leans negative — Fear + Extreme Fear = 48.7% of all days,
Greed + Extreme Greed = 36.2%. This reflects crypto's historical volatility
profile. For analysis, we may collapse to a binary `Fear` vs `Greed` bucket
(excluding Neutral) to get clean contrast groups. This decision will be made
in the cleaning notebook.

### Date Coverage

In [10]:
print('Date range:', fg['date'].min().date(), 'to', fg['date'].max().date())
print('Total days covered:', fg['date'].nunique())
print('Expected days (Feb 2018 – May 2025):', (fg['date'].max() - fg['date'].min()).days + 1)
print('Missing dates:', (fg['date'].max() - fg['date'].min()).days + 1 - fg['date'].nunique())

Date range: 2018-02-01 to 2025-05-02
Total days covered: 2644
Expected days (Feb 2018 – May 2025): 2648
Missing dates: 4


---
## Dataset 2 — Hyperliquid Trader Data

The trader table is the core of this project. Every downstream metric —
PnL, leverage proxy, win rate, long/short ratio — is derived from here.
We inspect it thoroughly before any cleaning or merging.

In [11]:
df = pd.read_csv('historical_data.csv')
df.shape

(211224, 16)

In [12]:
df.head(3)

,Account,Coin,Execution Price,Size Tokens,Size USD,Side,Timestamp IST,Start Position,Direction,Closed PnL,Transaction Hash,Order ID,Crossed,Fee,Trade ID,Timestamp
0,0xae5eacaf9c6b9111fd53034a602c192a04e082ed,@107,7.9769,986.87,7872.16,BUY,02-12-2024 22:50,0.000000,Buy,0.0,0xec09451986a1874e3a980418412fcd0201f500c95bac...,52017706630,True,0.345404,8.950000e+14,1.730000e+12
1,0xae5eacaf9c6b9111fd53034a602c192a04e082ed,@107,7.9800,16.00,127.68,BUY,02-12-2024 22:50,986.524596,Buy,0.0,0xec09451986a1874e3a980418412fcd0201f500c95bac...,52017706630,True,0.005600,4.430000e+14,1.730000e+12
2,0xae5eacaf9c6b9111fd53034a602c192a04e082ed,@107,7.9855,144.09,1150.63,BUY,02-12-2024 22:50,1002.518996,Buy,0.0,0xec09451986a1874e3a980418412fcd0201f500c95bac...,52017706630,True,0.050431,6.600000e+14,1.730000e+12


In [13]:
df.columns.tolist()

['Account',
 'Coin',
 'Execution Price',
 'Size Tokens',
 'Size USD',
 'Side',
 'Timestamp IST',
 'Start Position',
 'Direction',
 'Closed PnL',
 'Transaction Hash',
 'Order ID',
 'Crossed',
 'Fee',
 'Trade ID',
 'Timestamp']

In [14]:
df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 211224 entries, 0 to 211223
Data columns (total 16 columns):
 #   Column            Non-Null Count   Dtype  
---  ------            --------------   -----  
 0   Account           211224 non-null  object 
 1   Coin              211224 non-null  object 
 2   Execution Price   211224 non-null  float64
 3   Size Tokens       211224 non-null  float64
 4   Size USD          211224 non-null  float64
 5   Side              211224 non-null  object 
 6   Timestamp IST     211224 non-null  object 
 7   Start Position    211224 non-null  float64
 8   Direction         211224 non-null  object 
 9   Closed PnL        211224 non-null  float64
 10  Transaction Hash  211224 non-null  object 
 11  Order ID          211224 non-null  int64  
 12  Crossed           211224 non-null  bool   
 13  Fee               211224 non-null  float64
 14  Trade ID          211224 non-null  float64
 15  Timestamp         211224 non-null  float64
dtypes: bool(1), float64(

**Column Inventory — Trader Table:**

| Column | Type | ABSA Relevance |
|---|---|---|
| Account | str | Trader identity — segmentation key |
| Coin | str | Asset traded — filter/group by |
| Execution Price | float | Entry/exit price context |
| Size Tokens | float | Position size in token units |
| Size USD | float | Position size in USD — key metric |
| Side | str | BUY / SELL |
| Timestamp IST | str | Date-time — needs parsing |
| Start Position | float | Net position before this trade |
| Direction | str | Open Long / Close Short / etc — critical |
| Closed PnL | float | Realized profit/loss — primary metric |
| Transaction Hash | str | On-chain ref — audit only |
| Order ID | int | Order reference — dedup use |
| Crossed | bool | Taker (True) vs Maker (False) |
| Fee | float | Trading cost — net PnL adjustment |
| Trade ID | float | Trade reference |
| Timestamp | float | Unix ms — redundant with Timestamp IST |

### Missing Values & Duplicates

In [15]:
df.isnull().sum()

Account             0
Coin                0
Execution Price     0
Size Tokens         0
Size USD            0
Side                0
Timestamp IST       0
Start Position      0
Direction           0
Closed PnL          0
Transaction Hash    0
Order ID            0
Crossed             0
Fee                 0
Trade ID            0
Timestamp           0
dtype: int64

In [16]:
df.duplicated().sum()

np.int64(0)

**Missing Value Assessment — Trader Table:**

Zero nulls across all 16 columns. Zero exact-row duplicates.
This is structurally clean on arrival. The cleaning notebook will
focus on semantic issues — parsing timestamps, filtering trade types,
and engineering the leverage proxy — rather than null imputation.

### Accounts and Coverage

In [17]:
print('Unique accounts:', df['Account'].nunique())
print()
print('Trades per account:')
df['Account'].value_counts()

Unique accounts: 32

Trades per account:


Account
0xbee1707d6b44d4d52bfe19e41f8a828645437aab    40184
0xbaaaf6571ab7d571043ff1e313a9609a10637864    21192
0xa0feb3725a9335f49874d7cd8eaad6be45b27416    15605
0x8477e447846c758f5a675856001ea72298fd9cb5    14998
0xb1231a4a2dd02f2276fa3c5e2a2f3436e6bfed23    14733
0x28736f43f1e871e6aa8b1148d38d4994275d72c4    13311
0x513b8629fe877bb581bf244e326a047b249c4ff1    12236
0x75f7eeb85dc639d5e99c78f95393aa9a5f1170d4     9893
0x47add9a56df66b524d5e2c1993a43cde53b6ed85     8519
0x4f93fead39b70a1824f981a54d4e55b278e9f760     7584
0x23e7a7f8d14b550961925fbfdaa92f5d195ba5bd     7280
0xb899e522b5715391ae1d4f137653e7906c5e2115     4838
0x8170715b3b381dffb7062c0298972d4727a0a63b     4601
0x4acb90e786d897ecffb614dc822eb231b4ffb9f4     4356
0x083384f897ee0f19899168e3b1bec365f52a9012     3818
0x271b280974205ca63b716753467d5a371de622ab     3809
0x39cef799f8b69da1995852eea189df24eb5cae3c     3589
0x2c229d22b100a7beb69122eed721cee9b24011dd     3239
0x92f17e8d81a944691c10e753af1b1baae1a2cd0d     3052
0xbd

**Account Distribution Note:**

32 unique traders with highly unequal activity — the top account has 40184 trades
while the bottom has 332. This skew is expected in DeFi datasets where a small number
of active traders dominate volume. Segmentation by trade frequency will need
to account for this skew rather than use equal-size buckets.

### Date Coverage and Parsing

In [18]:
df['date'] = pd.to_datetime(df['Timestamp IST'], format='%d-%m-%Y %H:%M', errors='coerce').dt.date
print('Date range:', df['date'].min(), 'to', df['date'].max())
print('Unique trading days:', df['date'].nunique())
print('Null dates after parsing:', df['date'].isnull().sum())

Date range: 2023-05-01 to 2025-05-01
Unique trading days: 480
Null dates after parsing: 0


### Direction Categories

In [19]:
df['Direction'].value_counts()

Direction
Open Long                    49895
Close Long                   48678
Open Short                   39741
Close Short                  36013
Sell                         19902
Buy                          16716
Spot Dust Conversion           142
Short > Long                    70
Long > Short                    57
Auto-Deleveraging                8
Liquidated Isolated Short        1
Settlement                       1
Name: count, dtype: int64

**Direction Assessment:**

| Direction | Count | Role in Analysis |
|---|---|---|
| Open Long | 49,895 | Long entry — directional bias |
| Close Long | 48,678 | Long exit — PnL realized |
| Open Short | 39,741 | Short entry — directional bias |
| Close Short | 36,013 | Short exit — PnL realized |
| Buy | 16,716 | Spot buy trades — separate analysis|
| Sell | 19,902 | Spot sell trades — separate analysis |
| Other Rare Events| 336 | Spot Dust / Position flips / Auto-Deleveraging / Liquidations — track separately |

Perpetual futures dominate (87.6% of trades). The long/short open ratio
is ~1.26:1 overall — strong long bias in the sample, consistent with
a bull-market period (2023–2025). We will test whether this ratio
shifts meaningfully on Fear vs Greed days.

### PnL Distribution

In [20]:
df['Closed PnL'].describe()

count    211224.000000
mean         48.749001
std         919.164828
min     -117990.104100
25%           0.000000
50%           0.000000
75%           5.792797
max      135329.090100
Name: Closed PnL, dtype: float64

In [21]:
# PnL is only meaningful on closing trades
close_trades = df[df['Direction'].isin(['Close Long', 'Close Short'])]
print('Closing trades:', len(close_trades))
print()
print('PnL on closing trades:')
close_trades['Closed PnL'].describe()

Closing trades: 84691

PnL on closing trades:


count     84691.000000
mean         86.582157
std        1310.144468
min     -117990.104100
25%           0.428934
50%           5.881920
75%          36.962153
max      135329.090100
Name: Closed PnL, dtype: float64

In [22]:
# Win rate at trade level
wins = (close_trades['Closed PnL'] > 0).sum()
losses = (close_trades['Closed PnL'] < 0).sum()
zeros = (close_trades['Closed PnL'] == 0).sum()
print(f'Wins: {wins} ({wins/len(close_trades)*100:.1f}%)')
print(f'Losses: {losses} ({losses/len(close_trades)*100:.1f}%)')
print(f'Break-even: {zeros} ({zeros/len(close_trades)*100:.1f}%)')

Wins: 70754 (83.5%)
Losses: 13872 (16.4%)
Break-even: 65 (0.1%)


**PnL Distribution Assessment:**

The PnL distribution is heavily skewed, so comparing median and percentile values is more meaningful than using averages when analyzing performance under Fear vs Greed sentiment. The median PnL is `$ 5.88`, meaning a typical trade makes a small profit — most trades are modestly profitable. The mean PnL is `$86.58`, but this is much higher than the median because a few trades have extremely large wins or losses.
Trade outcomes range from - `$117,990 to +$135,329`, showing some very rare but extreme trades. Win rate is `83.5%`, with `16.4%` losses and almost none breaking even.

### Trade Size Distribution

In [23]:
df['Size USD'].describe()

count    2.112240e+05
mean     5.639451e+03
std      3.657514e+04
min      0.000000e+00
25%      1.937900e+02
50%      5.970450e+02
75%      2.058960e+03
max      3.921431e+06
Name: Size USD, dtype: float64

In [24]:
# Size distribution by percentile
for p in [10, 25, 50, 75, 90, 95, 99]:
    print(f'P{p}: ${df["Size USD"].quantile(p/100):,.0f}')

P10: $40
P25: $194
P50: $597
P75: $2,059
P90: $9,110
P95: $20,023
P99: $88,887


### Coin Distribution

In [25]:
print('Unique coins traded:', df['Coin'].nunique())
print()
print('Top 15 by trade count:')
df['Coin'].value_counts().head(15)

Unique coins traded: 246

Top 15 by trade count:


Coin
HYPE         68005
@107         29992
BTC          26064
ETH          11158
SOL          10691
FARTCOIN      4650
MELANIA       4428
PURR/USDC     2774
WLD           1983
SUI           1979
TRUMP         1920
XRP           1774
kPEPE         1730
kBONK         1647
FTT           1560
Name: count, dtype: int64

**Coin Distribution Note:**

HYPE dominates at 32% of all trades — this is Hyperliquid's native token,
reflecting strong home-field bias. BTC (12%) and ETH (5%) represent
the major blue chips. Tokens prefixed with `@` (e.g. `@107`, `@142`) are
internal Hyperliquid perpetual product codes. Analysis will be run on
the full dataset first, then validated on BTC/ETH only to check for coin-specific effects

---
## Dataset Alignment — Fear/Greed × Trader Data

In [26]:
trader_dates = set(df['date'].dropna().unique())
fg_dates = set(fg['date'].dt.date.unique())

overlap = trader_dates & fg_dates
trader_only = trader_dates - fg_dates

print(f'Trader data date range: {min(trader_dates)} to {max(trader_dates)}')
print(f'Fear/Greed date range:  {fg["date"].min().date()} to {fg["date"].max().date()}')
print()
print(f'Overlapping dates: {len(overlap)}')
print(f'Trader dates without sentiment: {len(trader_only)} → {trader_only}')

Trader data date range: 2023-05-01 to 2025-05-01
Fear/Greed date range:  2018-02-01 to 2025-05-02

Overlapping dates: 479
Trader dates without sentiment: 1 → {datetime.date(2024, 10, 26)}


In [27]:
# Test merge
fg_daily = fg[['date', 'value', 'classification']].copy()
fg_daily['date'] = fg_daily['date'].dt.date

merged = df.merge(fg_daily, on='date', how='left')
print('Rows after merge:', len(merged))
print('Null sentiment after merge:', merged['classification'].isnull().sum())
print()
print('Sentiment distribution in trader data:')
print(merged['classification'].value_counts())
print()
print('Sentiment distribution (%):')
print(merged['classification'].value_counts(normalize=True).round(3))

Rows after merge: 211224
Null sentiment after merge: 6

Sentiment distribution in trader data:
classification
Fear             61837
Greed            50303
Extreme Greed    39992
Neutral          37686
Extreme Fear     21400
Name: count, dtype: int64

Sentiment distribution (%):
classification
Fear             0.293
Greed            0.238
Extreme Greed    0.189
Neutral          0.178
Extreme Fear     0.101
Name: proportion, dtype: float64


**Alignment Assessment:**

Only 1 trader date (out of 480) has no matching sentiment reading — a 99.8% join
success rate. This is production-quality alignment. The 6 trades with null
sentiment will be dropped in cleaning (negligible loss).

**Sentiment distribution across trader data:**
- Fear days: 61,837 trades (29.3%)
- Greed days: 50,303 trades (23.8%)
- Neutral days: 37,686 trades (17.8%)
- Extreme Greed: 39,992 trades (18.9%)
- Extreme Fear: 21,400 trades (10.1%)

All five buckets are well-represented. The analysis will use both the
5-class breakdown and a simplified binary (Fear/Greed) contrast.

---
## Data Understanding Summary

### 1. Dataset Structure

Two tables joined on `date`:

| Table | Rows | Columns | Purpose |
|---|---|---|---|
| fear_greed_index.csv | 2,644 | 4 | Daily macro sentiment signal |
| historical_data.csv | 211224 | 16 | Trade-level execution data |

### 2. Fear/Greed Table

- 2,644 daily readings, Feb 2018 – May 2025
- Zero nulls, zero duplicates — textbook clean
- 5 classification buckets: Extreme Fear → Extreme Greed
- Fear dominates historically (48.7% of days)

### 3. Trader Table

- 211,224 trade records across 16 unique accounts
- Coverage: May 2023 – May 2025
- 246 unique coins; HYPE (32%), BTC (12%), ETH (5%) dominate
- Zero nulls, zero row-level duplicates
- Closing trades: 4,691 — primary PnL signal
- Strong long bias: Open Long / Open Short ≈ 1.4:1

### 4. Dataset Risks & Biases

| Risk | Detail | Management |
|---|---|---|
| Small account pool | Only 16 accounts | Treat as case study, not population |
| Long bias | 1.4:1 long/short | Normalize by regime, not absolute |
| HYPE concentration | 332% of trades | Validate findings on BTC/ETH subset |
| PnL skew | Max trade `+$135k`, min `-$117k` | Use median, not mean, for regime comparisons |
| Bull market period | 2023–2025 trends up | Fear regime findings may be regime-specific |

### 5. Analysis Viability Verdict

- 480 unique trading days with full sentiment labeling
- All 5 sentiment buckets represented in trader data
- 84,691 closing trades for PnL analysis
- Sufficient accounts for 3-way segmentation (leverage / frequency / consistency)
- 99.8% join success rate between datasets

**This dataset is viable for sentiment-driven behavioral analysis.**  
Proceeding to Notebook 02 — Data Extraction.